# LeetCode 1100 — Find K-Length Substrings With No Repeated Characters bằng Python

Link đề: https://leetcode.com/problems/find-k-length-substrings-with-no-repeated-characters/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sliding Window

> **Premium:** đây là bài **Premium** (🔒). Đề trong notebook được viết lại theo hiểu biết và theo bản mô tả trong kho tham khảo; nếu bạn có tài khoản Premium hãy **đối chiếu lại** đề gốc (đặc biệt: đếm **số vị trí** cửa sổ độ dài `k` có mọi ký tự đôi một khác nhau, hai cửa sổ trùng nội dung vẫn tính **hai lần**).

Skeleton LeetCode:

```python
class Solution:
    def numKLenSubstrNoRepeats(self, s: str, k: int) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**: số chuỗi con **liên tiếp** độ dài đúng `k` mà **không có ký tự nào lặp**. Điểm cốt lõi: **cửa sổ trượt** độ dài `k` trên chuỗi chữ thường (chỉ `26` chữ cái) — duy trì số ký tự bị lặp trong cửa sổ; nếu `k > 26` thì đáp án chắc chắn `0`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm các chuỗi con độ dài `k` (theo **vị trí**) mà mọi ký tự khác nhau.
2. Nắm công cụ: cửa sổ trượt cố định, mảng đếm `26` chữ cái, "số chữ cái đang lặp" (`dup`), vị trí xuất hiện gần nhất (`last seen`), tập hợp có thu nhỏ cửa sổ, mảng tổng tiền tố theo chữ cái.
3. Đi từ **kiểm tra từng cửa sổ bằng `set`** → **cửa sổ trượt có bộ đếm trùng** → **độ dài đoạn không lặp dài nhất kết thúc tại `i`** → **hai con trỏ + tập** → **tiền tố theo chữ cái**.
4. Hiểu mấu chốt: cửa sổ độ dài `k` kết thúc tại `i` hợp lệ ⇔ **đoạn không lặp dài nhất kết thúc tại `i` có độ dài `≥ k`**; mỗi bước trượt chỉ thêm một ký tự và bỏ một ký tự: `O(1)`.
5. Tránh lỗi kinh điển: bỏ sót cửa sổ cuối, dùng `set` khi trượt (xoá nhầm ký tự còn lặp), quên giảm bộ đếm trùng, đếm số chuỗi **khác nhau** thay vì số **vị trí**, nhảy `k` bước sau khi tìm thấy.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và số nguyên `k`. Trả về **số chuỗi con** của `s` có độ dài `k` và **không có ký tự lặp**.

**Ví dụ 1:** `s = "havefunonleetcode"`, `k = 5` → `6` (các chuỗi con `"havef"`, `"avefu"`, `"vefun"`, `"efuno"`, `"etcod"`, `"tcode"`).

**Ví dụ 2:** `s = "home"`, `k = 5` → `0` (`k` lớn hơn độ dài `s`: không có chuỗi con nào).

**Ràng buộc:** `1 <= s.length <= 10^4`; `s` chỉ gồm chữ cái thường; `1 <= k <= 10^4`.

In [1]:
import itertools
import random
import string
import time
from typing import List

examples = [("havefunonleetcode", 5, 6), ("home", 5, 0)]
for s, k, want in examples:
    print(f"s = {s!r}, k = {k} -> {want}")

s = 'havefunonleetcode', k = 5 -> 6
s = 'home', k = 5 -> 0


## 3. Kiến thức nền

### 3.1 Cửa sổ trượt độ dài cố định

Các chuỗi con độ dài `k` bắt đầu tại `i = 0, 1, …, n - k` (`n - k + 1` cửa sổ, nếu `k > n` thì không có). Cửa sổ tiếp theo bỏ ký tự đầu `s[i]` và thêm ký tự cuối `s[i + k]`.

### 3.2 Bộ đếm số chữ cái bị lặp

Giữ `cnt[c]` = số lần `c` trong cửa sổ và `dup` = **số chữ cái có `cnt ≥ 2`**. Cửa sổ không có ký tự lặp ⇔ `dup == 0`. Thêm `c`: `cnt[c] += 1`; nếu `cnt[c]` vừa thành `2` thì `dup += 1`. Bỏ `c`: nếu `cnt[c]` đang là `2` thì `dup -= 1`; rồi `cnt[c] -= 1`.

### 3.3 Chỉ có 26 chữ cái

Cửa sổ có `k > 26` ký tự **chắc chắn** có ký tự lặp (nguyên lý chuồng bồ câu) nên đáp án `0`. Vì vậy `k` lớn không gây khó khăn thêm.

In [2]:
def windows(s, k):
    """Mọi cửa sổ độ dài k theo vị trí (không có nếu k > len(s))."""
    return [s[i:i + k] for i in range(len(s) - k + 1)]


print(windows("havefunonleetcode", 5)[:4], windows("home", 5))
assert windows("home", 5) == [] and len(windows("havefunonleetcode", 5)) == 13

['havef', 'avefu', 'vefun', 'efuno'] []


## 4. Ý tưởng nền tảng / trực giác

Trượt một khung kính độ dài `k` dọc theo chuỗi và hỏi "trong khung có chữ nào lặp không?". Nếu mỗi lần trượt phải kiểm tra lại cả khung tốn `O(k)`; nhưng khung chỉ **đổi hai ký tự** (một ra, một vào), nên cập nhật bộ đếm và biết ngay có lặp hay không trong `O(1)`.

## 5. Brute force — kiểm tra từng cửa sổ bằng `set`

**Ý tưởng:** với mỗi cửa sổ `s[i:i+k]`, đúng khi `len(set(window)) == k` (mọi ký tự khác nhau ⇔ số ký tự khác nhau bằng độ dài). Đúng theo định nghĩa, làm **oracle**.

In [3]:
def count_brute(s, k):
    return sum(1 for w in windows(s, k) if len(set(w)) == k)


for s, k, want in examples:
    assert count_brute(s, k) == want
print([count_brute(s, k) for s, k, _ in examples])

[6, 0]


In [4]:
# Code nộp LeetCode cho Brute force (O(n · k); k tới 10^4 nhưng set rút gọn: vẫn qua vì cửa sổ > 26 ký tự luôn có lặp)
class SolutionBrute:
    def numKLenSubstrNoRepeats(self, s: str, k: int) -> int:
        count = 0
        for i in range(len(s) - k + 1):
            if len(set(s[i:i + k])) == k:
                count += 1
        return count

**Độ phức tạp:** `O(n · k)` thời gian (mỗi cửa sổ tạo `set` `O(k)`; chỉ chạy tốt vì `k ≤ n` và có thể cắt khi `k > 26`), `O(k)` bộ nhớ.

**Nút thắt:** mỗi cửa sổ kế tiếp chia sẻ `k - 1` ký tự với cửa sổ trước nhưng ta xây lại từ đầu.

## 6. Bước đột phá

Duy trì cửa sổ bằng bộ đếm `26` chữ cái và số chữ cái bị lặp `dup`: mỗi bước trượt `O(1)`. Tổng `O(n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** cửa sổ trượt với bộ đếm trùng (đáp án chính), **Cách 3** độ dài đoạn không lặp kết thúc tại `i` (vị trí gần nhất), **Cách 4** hai con trỏ + tập, **Cách 5** mảng tổng tiền tố theo chữ cái.

### Cách 2 — Cửa sổ trượt với bộ đếm trùng (đáp án chính)

**Giải thích:** thêm `s[i]`; khi `i ≥ k` bỏ `s[i - k]`; khi cửa sổ đủ `k` ký tự (`i ≥ k - 1`) và `dup == 0` thì tăng đáp án.

In [5]:
def count_window(s, k):
    if k > 26:
        return 0                                          # nguyên lý chuồng bồ câu
    cnt = [0] * 26
    dup = 0                                               # số chữ cái xuất hiện >= 2 lần trong cửa sổ
    ans = 0
    for i, ch in enumerate(s):
        c = ord(ch) - 97
        cnt[c] += 1
        if cnt[c] == 2:
            dup += 1
        if i >= k:                                        # bỏ ký tự rời khỏi cửa sổ
            o = ord(s[i - k]) - 97
            if cnt[o] == 2:
                dup -= 1
            cnt[o] -= 1
        if i >= k - 1 and dup == 0:
            ans += 1
    return ans


for s, k, want in examples:
    assert count_window(s, k) == want
print([count_window(s, k) for s, k, _ in examples])

[6, 0]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionWindow:
    def numKLenSubstrNoRepeats(self, s: str, k: int) -> int:
        if k > 26:
            return 0
        cnt = [0] * 26
        dup = 0
        ans = 0
        for i, ch in enumerate(s):
            c = ord(ch) - 97
            cnt[c] += 1
            if cnt[c] == 2:
                dup += 1
            if i >= k:
                o = ord(s[i - k]) - 97
                if cnt[o] == 2:
                    dup -= 1
                cnt[o] -= 1
            if i >= k - 1 and dup == 0:
                ans += 1
        return ans

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ (`26` ô).

### Cách 3 — Độ dài đoạn không lặp kết thúc tại `i`

**Ý tưởng:** gọi `left(i)` là vị trí bắt đầu nhỏ nhất sao cho `s[left..i]` không có ký tự lặp (đoạn không lặp dài nhất kết thúc tại `i`). Khi gặp `s[i]` đã xuất hiện ở vị trí `last[s[i]] ≥ left` thì `left = last[s[i]] + 1`. Cửa sổ độ dài `k` kết thúc tại `i` hợp lệ ⇔ `i - left + 1 ≥ k`.

In [7]:
def count_last_seen(s, k):
    last = {}
    left = 0
    ans = 0
    for i, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1                           # đẩy đầu đoạn qua lần xuất hiện trước
        last[ch] = i
        if i - left + 1 >= k:                             # đoạn không lặp kết thúc tại i đủ dài
            ans += 1
    return ans


for s, k, want in examples:
    assert count_last_seen(s, k) == want
print([count_last_seen(s, k) for s, k, _ in examples])

[6, 0]


In [8]:
# Code nộp LeetCode cho Cách 3
class SolutionLastSeen:
    def numKLenSubstrNoRepeats(self, s: str, k: int) -> int:
        last = {}
        left = ans = 0
        for i, ch in enumerate(s):
            if ch in last and last[ch] >= left:
                left = last[ch] + 1
            last[ch] = i
            if i - left + 1 >= k:
                ans += 1
        return ans

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ (`≤ 26` khoá).

### Cách 4 — Hai con trỏ + tập hợp

**Ý tưởng:** mẫu quen thuộc của "chuỗi con không lặp dài nhất": với mỗi `right`, trong khi `s[right]` đã có trong tập thì bỏ `s[left]` và tăng `left`; rồi thêm `s[right]`. Cửa sổ `[left, right]` luôn không lặp; nếu `right - left + 1 ≥ k` thì có đúng một cửa sổ độ dài `k` hợp lệ kết thúc tại `right`.

In [9]:
def count_two_pointers(s, k):
    seen = set()
    left = 0
    ans = 0
    for right, ch in enumerate(s):
        while ch in seen:
            seen.remove(s[left])
            left += 1
        seen.add(ch)
        if right - left + 1 >= k:
            ans += 1
    return ans


for s, k, want in examples:
    assert count_two_pointers(s, k) == want
print([count_two_pointers(s, k) for s, k, _ in examples])

[6, 0]


In [10]:
# Code nộp LeetCode cho Cách 4
class SolutionTwoPointers:
    def numKLenSubstrNoRepeats(self, s: str, k: int) -> int:
        seen = set()
        left = ans = 0
        for right, ch in enumerate(s):
            while ch in seen:
                seen.remove(s[left])
                left += 1
            seen.add(ch)
            if right - left + 1 >= k:
                ans += 1
        return ans

**Độ phức tạp:** `O(n)` thời gian (mỗi ký tự vào và ra tập tối đa một lần), `O(1)` bộ nhớ.

### Cách 5 — Mảng tổng tiền tố theo chữ cái

**Ý tưởng:** `pre[c][i]` = số lần chữ `c` trong `s[:i]`. Cửa sổ `[i, i+k)` hợp lệ ⇔ với mọi chữ `c`: `pre[c][i+k] - pre[c][i] ≤ 1`. Kiểm tra `26` chữ mỗi cửa sổ: `O(26 n)`; không cần cập nhật trạng thái.

In [11]:
def count_prefix(s, k):
    n = len(s)
    if k > n or k > 26:
        return 0
    pre = [[0] * (n + 1) for _ in range(26)]
    for i, ch in enumerate(s):
        for c in range(26):
            pre[c][i + 1] = pre[c][i]
        pre[ord(ch) - 97][i + 1] += 1
    ans = 0
    for i in range(n - k + 1):
        if all(pre[c][i + k] - pre[c][i] <= 1 for c in range(26)):
            ans += 1
    return ans


for s, k, want in examples:
    assert count_prefix(s, k) == want
print([count_prefix(s, k) for s, k, _ in examples])

[6, 0]


In [12]:
# Code nộp LeetCode cho Cách 5
class SolutionPrefix:
    def numKLenSubstrNoRepeats(self, s: str, k: int) -> int:
        n = len(s)
        if k > n or k > 26:
            return 0
        pre = [[0] * (n + 1) for _ in range(26)]
        for i, ch in enumerate(s):
            for c in range(26):
                pre[c][i + 1] = pre[c][i]
            pre[ord(ch) - 97][i + 1] += 1
        ans = 0
        for i in range(n - k + 1):
            if all(pre[c][i + k] - pre[c][i] <= 1 for c in range(26)):
                ans += 1
        return ans

**Độ phức tạp:** `O(26 n)` thời gian, `O(26 n)` bộ nhớ.

## 8. TRACE từng bước

Trace **Cách 2** với `s = "abcabcbb"`, `k = 3` (cửa sổ hợp lệ khi `dup = 0`, `i ≥ 2`):

| `i` | thêm | bỏ | `cnt` (chỉ chữ có mặt) | `dup` | cửa sổ | hợp lệ? |
|---|---|---|---|---|---|---|
| 0 | a | | a:1 | 0 | — | chưa đủ `k` |
| 1 | b | | a:1 b:1 | 0 | — | chưa đủ |
| 2 | c | | a:1 b:1 c:1 | 0 | `abc` | có (1) |
| 3 | a | a (vị trí 0) | b:1 c:1 a:1 | 0 | `bca` | có (2) |
| 4 | b | b (vị trí 1) | c:1 a:1 b:1 | 0 | `cab` | có (3) |
| 5 | c | c (vị trí 2) | a:1 b:1 c:1 | 0 | `abc` | có (4) |
| 6 | b | a (vị trí 3) | b:2 c:1 | 1 | `bcb` | không (`b` lặp) |
| 7 | b | b (vị trí 4) | c:1 b:2 | 1 | `cbb` | không (`b` lặp; `cnt[b]` đang `3` nên chưa giảm `dup`) |

Hàm trace in chính xác từng bước (kể cả `cnt` và `dup`):

In [13]:
def trace_window(s, k):
    cnt = {}
    dup = 0
    ans = 0
    print(f"s = {s!r}, k = {k}")
    for i, ch in enumerate(s):
        cnt[ch] = cnt.get(ch, 0) + 1
        if cnt[ch] == 2:
            dup += 1
        removed = ""
        if i >= k:
            o = s[i - k]
            removed = f", bỏ {o!r}"
            if cnt[o] == 2:
                dup -= 1
            cnt[o] -= 1
        win = s[max(0, i - k + 1):i + 1]
        ok = i >= k - 1 and dup == 0
        ans += ok
        live = {c: v for c, v in cnt.items() if v}
        print(f"  i = {i}: thêm {ch!r}{removed} | cnt {live} | dup {dup} | cửa sổ {win!r} {'HỢP LỆ' if ok else ''}")
    print("  đáp án =", ans)
    return ans


assert trace_window("abcabcbb", 3) == 4
assert trace_window("havefunonleetcode", 5) == 6

s = 'abcabcbb', k = 3
  i = 0: thêm 'a' | cnt {'a': 1} | dup 0 | cửa sổ 'a' 
  i = 1: thêm 'b' | cnt {'a': 1, 'b': 1} | dup 0 | cửa sổ 'ab' 
  i = 2: thêm 'c' | cnt {'a': 1, 'b': 1, 'c': 1} | dup 0 | cửa sổ 'abc' HỢP LỆ
  i = 3: thêm 'a', bỏ 'a' | cnt {'a': 1, 'b': 1, 'c': 1} | dup 0 | cửa sổ 'bca' HỢP LỆ
  i = 4: thêm 'b', bỏ 'b' | cnt {'a': 1, 'b': 1, 'c': 1} | dup 0 | cửa sổ 'cab' HỢP LỆ
  i = 5: thêm 'c', bỏ 'c' | cnt {'a': 1, 'b': 1, 'c': 1} | dup 0 | cửa sổ 'abc' HỢP LỆ
  i = 6: thêm 'b', bỏ 'a' | cnt {'b': 2, 'c': 1} | dup 1 | cửa sổ 'bcb' 
  i = 7: thêm 'b', bỏ 'b' | cnt {'b': 2, 'c': 1} | dup 1 | cửa sổ 'cbb' 
  đáp án = 4
s = 'havefunonleetcode', k = 5
  i = 0: thêm 'h' | cnt {'h': 1} | dup 0 | cửa sổ 'h' 
  i = 1: thêm 'a' | cnt {'h': 1, 'a': 1} | dup 0 | cửa sổ 'ha' 
  i = 2: thêm 'v' | cnt {'h': 1, 'a': 1, 'v': 1} | dup 0 | cửa sổ 'hav' 
  i = 3: thêm 'e' | cnt {'h': 1, 'a': 1, 'v': 1, 'e': 1} | dup 0 | cửa sổ 'have' 
  i = 4: thêm 'f' | cnt {'h': 1, 'a': 1, 'v': 1, 'e': 1

Bảng trên khớp với các dòng do hàm trace in: đáp án `4` cho `"abcabcbb"`, `k = 3` (các cửa sổ `abc`, `bca`, `cab`, `abc`).

## 9. Code nộp LeetCode

Chọn **Cách 2** (cửa sổ trượt với bộ đếm trùng). Giữ nguyên `class Solution`, `numKLenSubstrNoRepeats(self, s, k)`:

In [14]:
class Solution:
    def numKLenSubstrNoRepeats(self, s: str, k: int) -> int:
        if k > 26:                                  # cửa sổ > 26 ký tự chắc chắn có lặp
            return 0
        cnt = [0] * 26
        dup = 0                                     # số chữ cái đang xuất hiện >= 2 lần trong cửa sổ
        ans = 0
        for i, ch in enumerate(s):
            c = ord(ch) - 97
            cnt[c] += 1
            if cnt[c] == 2:
                dup += 1
            if i >= k:                              # ký tự s[i - k] rời khỏi cửa sổ
                o = ord(s[i - k]) - 97
                if cnt[o] == 2:
                    dup -= 1
                cnt[o] -= 1
            if i >= k - 1 and dup == 0:             # cửa sổ đủ k ký tự và không có lặp
                ans += 1
        return ans


print([Solution().numKLenSubstrNoRepeats(s, k) for s, k, _ in examples])
assert all(Solution().numKLenSubstrNoRepeats(s, k) == w for s, k, w in examples)

[6, 0]


## 10. Vì sao đúng

**(a) Bất biến của bộ đếm.** Sau khi xử lý `i` (thêm `s[i]`, bỏ `s[i-k]` nếu `i ≥ k`), `cnt` là số lần xuất hiện của từng chữ trong `s[max(0, i-k+1) .. i]` và `dup` là số chữ có `cnt ≥ 2`. Thêm `c`: `cnt[c]` tăng từ `x` lên `x+1`; số chữ có `cnt ≥ 2` tăng thêm `1` đúng khi `x + 1 = 2`. Bỏ `o`: `cnt[o]` giảm từ `x` xuống `x−1`; số chữ có `cnt ≥ 2` giảm `1` đúng khi `x = 2`. Hai cập nhật này duy trì bất biến.

**(b) Điều kiện hợp lệ.** Khi `i ≥ k − 1` cửa sổ có đúng `k` ký tự `s[i-k+1..i]`. Nó không có ký tự lặp ⇔ không chữ nào có `cnt ≥ 2` ⇔ `dup = 0`. Mỗi `i ≥ k − 1` ứng với đúng một cửa sổ (vị trí bắt đầu `i − k + 1`), nên `ans` đếm theo **vị trí**.

**(c) `k > 26`.** Mọi chuỗi `k > 26` ký tự trên bảng chữ `26` chữ có ít nhất một chữ lặp ⇒ `0`. (Và `k > n` thì không có cửa sổ vì điều kiện `i ≥ k − 1` không bao giờ đúng.)

**(d) Cách 3.** `left(i)` = vị trí bắt đầu nhỏ nhất của đoạn không lặp kết thúc tại `i`: `left(i) = max(left(i−1), last[s[i]] + 1)`. Cửa sổ `[i−k+1, i]` không lặp ⇔ nó nằm trọn trong đoạn không lặp kết thúc tại `i` ⇔ `i − k + 1 ≥ left(i)` ⇔ `i − left(i) + 1 ≥ k`.

**(e) Cách 4.** Vòng `while` giữ tập `seen` đúng bằng `s[left..right]` không lặp: chính là đoạn không lặp dài nhất kết thúc tại `right`, giống `left(right)` ở (d).

**(f) Cách 5.** Cửa sổ không có chữ lặp ⇔ mọi chữ `c` xuất hiện `≤ 1` lần, `= pre[c][i+k] − pre[c][i]`.

**(g) Kiểm chứng bằng máy.** Mọi cách khớp brute trên mọi chuỗi nhỏ với mọi `k` (mục 12), và các cách nhanh khớp nhau trên chuỗi dài `10^4`.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. `set` từng cửa sổ | `O(n k)` | `O(k)` | oracle |
| 2. Cửa sổ + bộ đếm trùng | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Vị trí gần nhất | `O(n)` | `O(1)` | không cần xoá khỏi cửa sổ |
| 4. Hai con trỏ + tập | `O(n)` | `O(1)` | mẫu "chuỗi con không lặp" |
| 5. Tiền tố theo chữ cái | `O(26 n)` | `O(26 n)` | kiểm tra độc lập từng cửa sổ |

Đo thực tế (`n = 10^4`):

In [15]:
def bench(fn, s, k, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(s, k)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(1100)
rand26 = "".join(rng.choice(string.ascii_lowercase) for _ in range(10_000))
cyc = (string.ascii_lowercase * 400)[:10_000]
for label, s, k in [("ngẫu nhiên 26 chữ, k = 8", rand26, 8), ("tuần hoàn 26 chữ, k = 26", cyc, 26)]:
    print(label)
    for name, fn in [("brute", count_brute), ("window", count_window), ("last_seen", count_last_seen), ("two_pointers", count_two_pointers), ("prefix", count_prefix)]:
        print(f"  {name:13s}: {bench(fn, s, k) * 1000:8.2f} ms -> {fn(s, k)}")

ngẫu nhiên 26 chữ, k = 8
  brute        :     4.10 ms -> 3022
  window       :     2.09 ms -> 3022
  last_seen    :     1.17 ms -> 3022
  two_pointers :     1.86 ms -> 3022


  prefix       :    28.33 ms -> 3022
tuần hoàn 26 chữ, k = 26


  brute        :     5.57 ms -> 9975


  window       :     2.14 ms -> 9975
  last_seen    :     1.32 ms -> 9975


  two_pointers :     1.37 ms -> 9975


  prefix       :    34.66 ms -> 9975


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi chuỗi nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [16]:
IMPLS = {
    "window": count_window,
    "last_seen": count_last_seen,
    "two_pointers": count_two_pointers,
    "prefix": count_prefix,
    "Solution": lambda s, k: Solution().numKLenSubstrNoRepeats(s, k),
    "SolutionWindow": lambda s, k: SolutionWindow().numKLenSubstrNoRepeats(s, k),
    "SolutionLastSeen": lambda s, k: SolutionLastSeen().numKLenSubstrNoRepeats(s, k),
    "SolutionTwoPointers": lambda s, k: SolutionTwoPointers().numKLenSubstrNoRepeats(s, k),
    "SolutionPrefix": lambda s, k: SolutionPrefix().numKLenSubstrNoRepeats(s, k),
}
BRUTE = {"brute": count_brute, "SolutionBrute": lambda s, k: SolutionBrute().numKLenSubstrNoRepeats(s, k)}


def check(s, k, want=None, brute=True):
    if want is None:
        want = count_brute(s, k) if brute else Solution().numKLenSubstrNoRepeats(s, k)
    impls = dict(IMPLS)
    if brute:
        impls.update(BRUTE)
    for name, fn in impls.items():
        got = fn(s, k)
        assert got == want, (name, s, k, got, want)
    return want


for s, k, want in examples:
    check(s, k, want)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "cách nhanh +", len(BRUTE), "vét cạn")

ví dụ đề: mọi cài đặt đúng; 9 cách nhanh + 2 vét cạn


### B. Edge case

- `k = 1`: mọi vị trí hợp lệ ⇒ đáp án `n`.
- `k > n`: `0`; `k = n` và chuỗi toàn khác nhau: `1`; `k = n` có lặp: `0`.
- `k > 26`: `0` (kể cả khi `k ≤ n`).
- `k = 26` với đúng cả bảng chữ cái, và với bảng chữ cái dịch vòng.
- Toàn một chữ (`"aaaa"`) với `k = 1` và `k = 2`.
- Chuỗi tuần hoàn `"abcabc"` (các cửa sổ trùng nội dung vẫn tính hai lần).
- Ký tự lặp ở đầu và cuối cửa sổ (`"abca"`, `k = 4` → `0`; `k = 3` → `2`).

In [17]:
alpha = string.ascii_lowercase
edge = [
    ("a", 1, 1), ("a", 2, 0), ("ab", 1, 2), ("ab", 2, 1), ("aa", 2, 0), ("aaaa", 1, 4), ("aaaa", 2, 0),
    ("abcabc", 3, 4), ("abca", 4, 0), ("abca", 3, 2), ("abcdef", 6, 1), ("abcdef", 7, 0),
    (alpha, 26, 1), (alpha + "a", 26, 2), (alpha * 3, 26, 53), (alpha * 3, 27, 0), ("z" * 50, 1, 50),
    ("abcdefghij" * 3, 10, 21), ("abcdefghij" * 3, 11, 0),
]
for s, k, want in edge:
    got = check(s, k, want, brute=len(s) <= 100)
print(len(edge), "edge case: mọi cài đặt (kể cả vét cạn) khớp")

19 edge case: mọi cài đặt (kể cả vét cạn) khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_miss_last_window(s, k):                         # range thiếu +1: bỏ cửa sổ cuối
    return sum(1 for i in range(len(s) - k) if len(set(s[i:i + k])) == k)


def bug_set_slide(s, k):                                # trượt bằng set: xoá ký tự ra khỏi tập dù còn bản sao
    seen = set()
    ans = 0
    for i, ch in enumerate(s):
        if i >= k:
            seen.discard(s[i - k])
        seen.add(ch)
        if i >= k - 1 and len(seen) == k:
            ans += 1
    return ans


def bug_dup_never_decreases(s, k):                      # quên giảm dup khi ký tự lặp rời cửa sổ
    cnt = [0] * 26
    dup = 0
    ans = 0
    for i, ch in enumerate(s):
        c = ord(ch) - 97
        cnt[c] += 1
        if cnt[c] == 2:
            dup += 1
        if i >= k:
            cnt[ord(s[i - k]) - 97] -= 1
        if i >= k - 1 and dup == 0:
            ans += 1
    return ans


def bug_distinct_strings(s, k):                         # đếm số chuỗi KHÁC NHAU thay vì số vị trí
    return len({w for w in windows(s, k) if len(set(w)) == k})


def bug_non_overlapping(s, k):                          # tìm thấy rồi nhảy k bước
    i, ans = 0, 0
    while i + k <= len(s):
        if len(set(s[i:i + k])) == k:
            ans += 1
            i += k
        else:
            i += 1
    return ans


def bug_at_most_k(s, k):                                # đếm cửa sổ có số ký tự khác nhau >= k - 1 (lệch 1)
    return sum(1 for w in windows(s, k) if len(set(w)) >= k - 1)


BUGS = [bug_miss_last_window, bug_set_slide, bug_dup_never_decreases, bug_distinct_strings, bug_non_overlapping, bug_at_most_k]
pool = [("".join(t), k) for n in range(1, 8) for t in itertools.product("abc", repeat=n) for k in range(1, 5)]
for bug in BUGS:
    found = next(((s, k) for s, k in pool if bug(s, k) != count_brute(s, k)), None)
    assert found, bug.__name__
    s, k = found
    print(f"{bug.__name__:24s} sai trên s = {s!r}, k = {k}: ra {bug(s, k)}, đúng {count_brute(s, k)}")

bug_miss_last_window     sai trên s = 'a', k = 1: ra 0, đúng 1
bug_set_slide            sai trên s = 'aab', k = 2: ra 0, đúng 1
bug_dup_never_decreases  sai trên s = 'aa', k = 1: ra 1, đúng 2
bug_distinct_strings     sai trên s = 'aa', k = 1: ra 1, đúng 2
bug_non_overlapping      sai trên s = 'aba', k = 2: ra 1, đúng 2
bug_at_most_k            sai trên s = 'aa', k = 2: ra 1, đúng 0


### D. Vét cạn mọi chuỗi nhỏ

Mọi chuỗi trên `{a, b, c}` độ dài `1..8` (`9 840` chuỗi) và `{a, b, c, d}` độ dài `1..6` (`5 460` chuỗi), mỗi chuỗi với mọi `k = 1..n+2` (kể cả `k > n`). So mọi cài đặt với brute.

In [19]:
t0 = time.perf_counter()
cnt = 0
for alphabet, nmax in (("abc", 8), ("abcd", 6)):
    for n in range(1, nmax + 1):
        for tup in itertools.product(alphabet, repeat=n):
            s = "".join(tup)
            for k in range(1, n + 3):
                want = count_brute(s, k)
                for name, fn in IMPLS.items():
                    assert fn(s, k) == want, (name, s, k)
                cnt += 1
print(cnt, "cặp (s, k):", len(IMPLS), "cài đặt khớp brute;", round(time.perf_counter() - t0, 1), "s")

135360 cặp (s, k): 9 cài đặt khớp brute; 4.8 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Chuỗi ngẫu nhiên nhỏ: mọi cách khớp brute; (2) `k = 1 → n`; `k > 26 → 0`; `k > n → 0`; đáp án `≤ max(0, n − k + 1)`; (3) **đảo chuỗi** không đổi; (4) **hoán vị bảng chữ cái** không đổi; (5) **thêm một ký tự** vào cuối làm đáp án tăng `0` hoặc `1`; (6) chuỗi gồm `m ≤ 26` chữ cái **đôi một khác nhau**: đáp án `max(0, m − k + 1)`; (7) chuỗi tuần hoàn chu kỳ `p ≤ 26` toàn chữ khác nhau: `max(0, n − k + 1)` khi `k ≤ p`.

In [20]:
rng = random.Random(20240200)
t0 = time.perf_counter()
for it in range(1500):
    n = rng.randint(1, 60)
    alpha_n = rng.choice([2, 3, 5, 8, 26])
    s = "".join(rng.choice(string.ascii_lowercase[:alpha_n]) for _ in range(n))
    k = rng.randint(1, n + 3)
    ans = check(s, k)                                                                 # (1)
    assert ans <= max(0, n - k + 1)                                                   # (2)
    if k == 1:
        assert ans == n
    if k > 26 or k > n:
        assert ans == 0
    assert Solution().numKLenSubstrNoRepeats(s[::-1], k) == ans                       # (3)
    perm = list(string.ascii_lowercase)
    rng.shuffle(perm)
    tr = str.maketrans(string.ascii_lowercase, "".join(perm))
    assert Solution().numKLenSubstrNoRepeats(s.translate(tr), k) == ans               # (4)
    d = Solution().numKLenSubstrNoRepeats(s + rng.choice(string.ascii_lowercase), k) - ans
    assert d in (0, 1)                                                                # (5)
for m in range(1, 27):
    s = "".join(rng.sample(string.ascii_lowercase, m))
    for k in range(1, 30):
        assert Solution().numKLenSubstrNoRepeats(s, k) == max(0, m - k + 1)           # (6)
for p in range(1, 27):
    base = "".join(rng.sample(string.ascii_lowercase, p))
    s = (base * (100 // p + 2))[:100]
    for k in range(1, p + 1):
        assert Solution().numKLenSubstrNoRepeats(s, k) == max(0, 100 - k + 1)         # (7)
print("(1)-(7) 1500 chuỗi ngẫu nhiên + chuỗi cấu trúc:", round(time.perf_counter() - t0, 1), "s")

(1)-(7) 1500 chuỗi ngẫu nhiên + chuỗi cấu trúc: 0.3 s


### F. Giới hạn

`n = 10^4`: ngẫu nhiên `26` chữ; tuần hoàn `26` chữ (`k = 26 → n − 25`); toàn một chữ; hai chữ xen kẽ; `k = 10^4` (bằng `n`); `k = 27` (luôn `0`). Các cách nhanh khớp nhau.

In [21]:
t0 = time.perf_counter()
rg = random.Random(5)
N = 10_000
cases = [
    ("".join(rg.choice(string.ascii_lowercase) for _ in range(N)), 8),
    ("".join(rg.choice(string.ascii_lowercase) for _ in range(N)), 26),
    ((string.ascii_lowercase * 400)[:N], 26),
    ("a" * N, 1), ("a" * N, 2), ("ab" * (N // 2), 2), ("ab" * (N // 2), 3),
    ("".join(rg.choice(string.ascii_lowercase) for _ in range(N)), N), ((string.ascii_lowercase * 400)[:N], 27),
]
for s, k in cases:
    want = Solution().numKLenSubstrNoRepeats(s, k)
    for name, fn in IMPLS.items():
        if name == "prefix":
            continue                                       # O(26 n) bộ nhớ lớn: kiểm tra riêng ở dưới
        assert fn(s, k) == want, (name, k)
assert Solution().numKLenSubstrNoRepeats((string.ascii_lowercase * 400)[:N], 26) == N - 25
assert Solution().numKLenSubstrNoRepeats("a" * N, 1) == N and Solution().numKLenSubstrNoRepeats("ab" * (N // 2), 2) == N - 1
assert count_prefix(cases[0][0], 8) == Solution().numKLenSubstrNoRepeats(cases[0][0], 8)
print(len(cases), "test giới hạn n = 10^4:", round(time.perf_counter() - t0, 1), "s")

9 test giới hạn n = 10^4: 0.4 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: bỏ sót cửa sổ cuối

In [22]:
print("bug:", bug_miss_last_window("abc", 3), "| đúng:", Solution().numKLenSubstrNoRepeats("abc", 3))
assert bug_miss_last_window("abc", 3) == 0 and Solution().numKLenSubstrNoRepeats("abc", 3) == 1

bug: 0 | đúng: 1


**Vì sao sai:** có `n - k + 1` cửa sổ (vị trí bắt đầu `0..n-k`). `range(len(s) - k)` bỏ mất cửa sổ cuối.

### Lỗi 2: trượt bằng `set` (xoá nhầm ký tự còn lặp)

In [23]:
s, k = next((s, k) for s, k in pool if bug_set_slide(s, k) != count_brute(s, k))
print("phản ví dụ:", repr(s), k, "| bug:", bug_set_slide(s, k), "| đúng:", count_brute(s, k))
assert bug_set_slide(s, k) != count_brute(s, k)

phản ví dụ: 'aab' 2 | bug: 0 | đúng: 1


**Vì sao sai:** `set` không nhớ số lần: khi một chữ xuất hiện hai lần trong cửa sổ, `set` chỉ giữ một bản; lúc một bản rời đi, `discard` xoá chữ đó dù bản còn lại vẫn ở trong cửa sổ. Phải dùng **bộ đếm**.

### Lỗi 3: quên giảm bộ đếm trùng

In [24]:
s, k = next((s, k) for s, k in pool if bug_dup_never_decreases(s, k) != count_brute(s, k))
print("phản ví dụ:", repr(s), k, "| bug:", bug_dup_never_decreases(s, k), "| đúng:", count_brute(s, k))
assert bug_dup_never_decreases(s, k) < count_brute(s, k)

phản ví dụ: 'aa' 1 | bug: 1 | đúng: 2


**Vì sao sai:** khi ký tự lặp rời cửa sổ, `dup` phải giảm; nếu không, mọi cửa sổ về sau bị coi là còn lặp.

### Lỗi 4: đếm chuỗi khác nhau thay vì số vị trí

In [25]:
print("bug:", bug_distinct_strings("abcabc", 3), "| đúng:", Solution().numKLenSubstrNoRepeats("abcabc", 3))
assert bug_distinct_strings("abcabc", 3) == 3 and Solution().numKLenSubstrNoRepeats("abcabc", 3) == 4

bug: 3 | đúng: 4


**Vì sao sai:** đề đếm **số chuỗi con** theo vị trí (hai lần `"abc"` ở hai vị trí khác nhau tính hai); không khử trùng nội dung. (Ví dụ 1 cũng cho thấy: `"havef"` ... các cửa sổ đều khác nhau nên không phân biệt được — kiểm tra bằng `"abcabc"`.)

### Lỗi 5: nhảy `k` bước sau khi tìm thấy

In [26]:
print("bug:", bug_non_overlapping("abcabc", 3), "| đúng:", Solution().numKLenSubstrNoRepeats("abcabc", 3))
assert bug_non_overlapping("abcabc", 3) == 2

bug: 2 | đúng: 4


**Vì sao sai:** các cửa sổ được phép **chồng lấn** (`"abc"`, `"bca"`, `"cab"`, `"abc"`); không nhảy `k`.

### Lỗi 6 (bonus): lệch `1` ở điều kiện số ký tự khác nhau

In [27]:
s, k = next((s, k) for s, k in pool if bug_at_most_k(s, k) != count_brute(s, k))
print("phản ví dụ:", repr(s), k, "| bug:", bug_at_most_k(s, k), "| đúng:", count_brute(s, k))
assert bug_at_most_k(s, k) > count_brute(s, k)

phản ví dụ: 'aa' 2 | bug: 1 | đúng: 0


**Vì sao sai:** cần **đúng `k`** ký tự khác nhau (`len(set(w)) == k`), tức mọi ký tự đôi một khác nhau; `≥ k − 1` chấp nhận một cặp trùng.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đếm chuỗi con **độ dài cố định `k`** thoả tính chất về tần suất ký tự" → **cửa sổ trượt cố định** với bộ đếm; "không lặp" → theo dõi số chữ đang lặp.

**Mẹo:**

1. `k > 26` ⇒ `0` (bảng chữ cái chỉ có `26` chữ).
2. Thêm ký tự mới trước, bỏ ký tự cũ sau; kiểm tra khi `i ≥ k - 1`.
3. Dùng `dup` = số chữ có `cnt ≥ 2` để kiểm tra `O(1)`.
4. Cách khác: độ dài đoạn không lặp kết thúc tại `i` `≥ k`.
5. Đếm theo **vị trí**, không khử trùng nội dung.
6. Oracle: `len(set(window)) == k` cho mọi cửa sổ.

**Họ bài cùng pattern:** Longest Substring Without Repeating Characters (3), Permutation in String (567), Find All Anagrams in a String (438), Maximum Number of Vowels in a Substring of Given Length (1456), Number of Substrings Containing All Three Characters (1358), Substrings of Size Three with Distinct Characters (1876).

## 15. Biến thể

1. **Chuỗi con không lặp dài nhất** (LeetCode 3): `max(i - left + 1)`.
2. **Số chuỗi con mọi độ dài không lặp:** `Σ_i (i - left(i) + 1)`.
3. **Cửa sổ có đúng `k` chữ khác nhau** (không nhất thiết `k` ký tự).
4. **Liệt kê** các cửa sổ hợp lệ.

Cài đặt và đối chiếu brute:

In [28]:
def longest_no_repeat(s):
    last, left, best = {}, 0, 0
    for i, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1
        last[ch] = i
        best = max(best, i - left + 1)
    return best


def count_all_no_repeat(s):
    last, left, total = {}, 0, 0
    for i, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1
        last[ch] = i
        total += i - left + 1                             # mọi đoạn không lặp kết thúc tại i
    return total


def windows_with_distinct(s, k, d):
    cnt, distinct, ans = {}, 0, 0
    for i, ch in enumerate(s):
        cnt[ch] = cnt.get(ch, 0) + 1
        if cnt[ch] == 1:
            distinct += 1
        if i >= k:
            o = s[i - k]
            cnt[o] -= 1
            if cnt[o] == 0:
                distinct -= 1
        if i >= k - 1 and distinct == d:
            ans += 1
    return ans


def list_valid_windows(s, k):
    return [s[i:i + k] for i in range(len(s) - k + 1) if len(set(s[i:i + k])) == k]


rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 12)
    s = "".join(rng.choice("abcd") for _ in range(n))
    assert longest_no_repeat(s) == max(j - i for i in range(n) for j in range(i, n + 1) if len(set(s[i:j])) == j - i)
    assert count_all_no_repeat(s) == sum(1 for i in range(n) for j in range(i + 1, n + 1) if len(set(s[i:j])) == j - i)
    k = rng.randint(1, n + 1)
    for d in range(1, 5):
        assert windows_with_distinct(s, k, d) == sum(1 for w in windows(s, k) if len(set(w)) == d)
    assert len(list_valid_windows(s, k)) == Solution().numKLenSubstrNoRepeats(s, k)
print("đoạn không lặp dài nhất / số đoạn không lặp / cửa sổ có đúng d chữ khác nhau / liệt kê: khớp brute")

đoạn không lặp dài nhất / số đoạn không lặp / cửa sổ có đúng d chữ khác nhau / liệt kê: khớp brute


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `set` từng cửa sổ | định nghĩa | `O(n k)` | `O(k)` | oracle |
| 2. Cửa sổ + `dup` | trượt `O(1)` | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Vị trí gần nhất | độ dài đoạn không lặp | `O(n)` | `O(1)` | không cần "bỏ" ký tự |
| 4. Hai con trỏ + tập | mẫu bài chuỗi con không lặp | `O(n)` | `O(1)` | quen thuộc |
| 5. Tiền tố theo chữ | kiểm tra độc lập | `O(26 n)` | `O(26 n)` | đơn giản, không trạng thái |

Mọi cách khớp brute trên `15 300` chuỗi nhỏ nhân mọi `k` và `1 500` chuỗi ngẫu nhiên; khớp nhau trên `n = 10^4`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().numKLenSubstrNoRepeats("abcd", 2), Solution().numKLenSubstrNoRepeats("aab", 2), Solution().numKLenSubstrNoRepeats("abab", 2), Solution().numKLenSubstrNoRepeats("abcabc", 4))

3 1 3 0


### Bài 2 (tính tay)

`s = "abcbde"`, `k = 3`. Liệt kê các cửa sổ độ dài `3` và đánh dấu hợp lệ. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cửa sổ trượt với bộ đếm trùng:

In [30]:
def count_todo(s, k):
    if k > 26:
        return 0
    cnt = [0] * 26
    dup = ans = 0
    for i, ch in enumerate(s):
        c = ord(ch) - 97
        pass            # TODO 1: cnt[c] += 1; nếu cnt[c] == 2 thì dup += 1
        pass            # TODO 2: nếu i >= k: bỏ s[i - k] (nếu cnt == 2 thì dup -= 1; rồi cnt -= 1)
        pass            # TODO 3: nếu i >= k - 1 và dup == 0 thì ans += 1
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(count_todo(s, k) == w for s, k, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **vị trí xuất hiện gần nhất** (`last seen`):

In [31]:
class MySolution:
    def numKLenSubstrNoRepeats(self, s, k):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().numKLenSubstrNoRepeats(s, k) == w for s, k, w in examples)

### Bài 5 (mini challenge): số chuỗi con mọi độ dài không lặp

Viết `count_all(s)` trả về số chuỗi con (theo vị trí) **mọi độ dài `≥ 1`** không có ký tự lặp. Ví dụ `count_all("abc") == 6`, `count_all("aab") == 4` (`a`, `a`, `b`, `ab`).

In [32]:
def count_all(s):
    pass            # TODO


# Khi làm xong:
# assert count_all("abc") == 6 and count_all("aab") == 4 and count_all("aaa") == 3

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đếm cửa sổ độ dài `k` có mọi ký tự khác nhau.
- Mức 2: `"abab"`, `k = 2`: các cửa sổ `ab`, `ba`, `ab` đều hợp lệ; `"abcabc"`, `k = 4`: `abca` có lặp.
- Mức 3: `3 1 3 0`.

**Bài 2**

- Mức 1: cửa sổ: `abc`, `bcb`, `cbd`, `bde`.
- Mức 2: `bcb` có `b` lặp.
- Mức 3: đáp án `3`.

**Bài 3**

- Mức 1: `cnt[c] += 1`; `if cnt[c] == 2: dup += 1`.
- Mức 2: `if i >= k: o = ord(s[i-k]) - 97; if cnt[o] == 2: dup -= 1; cnt[o] -= 1`.
- Mức 3: `if i >= k - 1 and dup == 0: ans += 1`.

**Bài 4**

- Mức 1: `last = {}`, `left = 0`.
- Mức 2: nếu `ch in last and last[ch] >= left`: `left = last[ch] + 1`; sau đó `last[ch] = i`.
- Mức 3: `if i - left + 1 >= k: ans += 1`.

**Bài 5**

- Mức 1: số đoạn không lặp kết thúc tại `i` bằng độ dài đoạn không lặp dài nhất kết thúc tại `i`.
- Mức 2: cộng `i - left + 1` ở mỗi `i`.
- Mức 3: `count_all("aab")`: `i=0 → 1`, `i=1 → 1` (left = 1), `i=2 → 2` ⇒ `4`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().numKLenSubstrNoRepeats("abcd", 2), Solution().numKLenSubstrNoRepeats("aab", 2), Solution().numKLenSubstrNoRepeats("abab", 2), Solution().numKLenSubstrNoRepeats("abcabc", 4))
print(vals)
assert vals == (3, 1, 3, 0) == (count_brute("abcd", 2), count_brute("aab", 2), count_brute("abab", 2), count_brute("abcabc", 4))
print("Bài 1 OK")

(3, 1, 3, 0)
Bài 1 OK


### Bài 2

In [34]:
ws = windows("abcbde", 3)
print(ws, [len(set(w)) == 3 for w in ws])
assert ws == ["abc", "bcb", "cbd", "bde"] and Solution().numKLenSubstrNoRepeats("abcbde", 3) == 3
trace_window("abcbde", 3)
print("Bài 2 OK")

['abc', 'bcb', 'cbd', 'bde'] [True, False, True, True]
s = 'abcbde', k = 3
  i = 0: thêm 'a' | cnt {'a': 1} | dup 0 | cửa sổ 'a' 
  i = 1: thêm 'b' | cnt {'a': 1, 'b': 1} | dup 0 | cửa sổ 'ab' 
  i = 2: thêm 'c' | cnt {'a': 1, 'b': 1, 'c': 1} | dup 0 | cửa sổ 'abc' HỢP LỆ
  i = 3: thêm 'b', bỏ 'a' | cnt {'b': 2, 'c': 1} | dup 1 | cửa sổ 'bcb' 
  i = 4: thêm 'd', bỏ 'b' | cnt {'b': 1, 'c': 1, 'd': 1} | dup 0 | cửa sổ 'cbd' HỢP LỆ
  i = 5: thêm 'e', bỏ 'c' | cnt {'b': 1, 'd': 1, 'e': 1} | dup 0 | cửa sổ 'bde' HỢP LỆ
  đáp án = 3
Bài 2 OK


### Bài 3

In [35]:
def count_answer(s, k):
    if k > 26:
        return 0
    cnt = [0] * 26
    dup = ans = 0
    for i, ch in enumerate(s):
        c = ord(ch) - 97
        cnt[c] += 1
        if cnt[c] == 2:
            dup += 1
        if i >= k:
            o = ord(s[i - k]) - 97
            if cnt[o] == 2:
                dup -= 1
            cnt[o] -= 1
        if i >= k - 1 and dup == 0:
            ans += 1
    return ans


assert all(count_answer(s, k) == w for s, k, w in examples)
for s, k in pool:
    assert count_answer(s, k) == count_brute(s, k)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def numKLenSubstrNoRepeats(self, s, k):
        last, left, ans = {}, 0, 0
        for i, ch in enumerate(s):
            if ch in last and last[ch] >= left:
                left = last[ch] + 1
            last[ch] = i
            if i - left + 1 >= k:
                ans += 1
        return ans


assert all(MySolutionAnswer().numKLenSubstrNoRepeats(s, k) == w for s, k, w in examples)
for s, k in pool:
    assert MySolutionAnswer().numKLenSubstrNoRepeats(s, k) == count_brute(s, k)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def count_all(s):
    return count_all_no_repeat(s)


assert count_all("abc") == 6 and count_all("aab") == 4 and count_all("aaa") == 3
for s, _ in pool[:2000]:
    n = len(s)
    assert count_all(s) == sum(1 for i in range(n) for j in range(i + 1, n + 1) if len(set(s[i:j])) == j - i)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm cửa sổ độ dài `k` không có ký tự lặp):**

```python
class Solution:
    def numKLenSubstrNoRepeats(self, s: str, k: int) -> int:
        if k > 26:
            return 0
        cnt = [0] * 26
        dup = ans = 0
        for i, ch in enumerate(s):
            c = ord(ch) - 97
            cnt[c] += 1
            if cnt[c] == 2:
                dup += 1
            if i >= k:
                o = ord(s[i - k]) - 97
                if cnt[o] == 2:
                    dup -= 1
                cnt[o] -= 1
            if i >= k - 1 and dup == 0:
                ans += 1
        return ans
```

**Checklist trước khi submit:**

- [ ] `k > 26` hoặc `k > n` trả `0`?
- [ ] Có `n - k + 1` cửa sổ (không bỏ cửa sổ cuối)?
- [ ] Dùng **bộ đếm** (không phải `set`) khi trượt, và giảm `dup` đúng lúc?
- [ ] Đếm theo **vị trí** (cửa sổ trùng nội dung vẫn tính riêng)?
- [ ] Đã thử `k = 1`, `k = n`, `"abcabc"`, `"aaaa"`, bảng chữ cái đủ `26`?
- [ ] Đúng tên `class Solution`, `numKLenSubstrNoRepeats(self, s, k)`, không in debug.